# The Trade Finance Doc Mismatch Detector, in one notebook

A runnable miniature of the pipeline in [`Detector/`](Detector/): three document
families instead of eight, one file instead of a package, but the same shape.
Classify every presented document, extract it under the schema its classification
selected, then reconcile the whole presentation against UCP 600.

[`WALKTHROUGH.md`](WALKTHROUGH.md) narrates the production code stage by stage.
This notebook builds a working version of it from scratch, top to bottom.

**To run the live cell at the end**, set a Gemini API key first:

```bash
export GEMINI_API_KEY=...
```

Without a key everything up to §14 still runs — the agents are built on
pydantic-ai's `test` model — and §14 skips itself.

The OCR step in §11.2 reads documents with Amazon Textract, so `aioboto3` has to be
installed:

```bash
uv add aioboto3
```

AWS credentials are optional. A document that arrives as text skips Textract
entirely, and §15 runs the OCR fan-out against a stub — no account, no spend.

## 1. Setup

Standard library and Pydantic only. pydantic-ai does not appear until §9, because
the first eight sections are about the *types*, and the types are what the model
is eventually asked to fill in.

In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal
from enum import StrEnum
from typing import Annotated, Any, Literal, cast

from pydantic import BaseModel, ConfigDict, Field

## 2. Enumerations

Three closed vocabularies: which family a document belongs to, how badly a finding
hurts, and the verdict on the case. `StrEnum` so they compare equal to their wire
strings and survive a JSON round trip.

In [2]:
class DocumentType(StrEnum):
    """The document families this miniature understands (the real one has 8)."""

    LETTER_OF_CREDIT = 'letter_of_credit'
    COMMERCIAL_INVOICE = 'commercial_invoice'
    BILL_OF_LADING = 'bill_of_lading'
    UNKNOWN = 'unknown'


class Severity(StrEnum):
    """How badly a discrepancy hurts the presentation."""

    CRITICAL = 'critical'
    """A documentary discrepancy under LC rules: the bank would refuse."""

    WARNING = 'warning'
    """An ambiguity a human examiner should look at."""

    INFO = 'info'
    """A formatting difference or informational observation."""


class CaseStatus(StrEnum):
    """The overall verdict for a presentation."""

    CLEAN = 'clean'
    NEEDS_REVIEW = 'needs_review'
    BLOCKED = 'blocked'

## 3. Extraction payloads — the schema *is* the prompt

One model per document family. pydantic-ai sends each model's JSON schema to the
extractor as the output contract, field descriptions included — so a `description=`
here is an instruction the model actually reads, not a comment for the next
developer. `extra='forbid'` closes the schema; `use_attribute_docstrings=True`
promotes the docstring under an attribute into its description.

In [3]:
class ExtractionBase(BaseModel):
    """Shared configuration for every extraction payload."""

    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)


class LetterOfCredit(ExtractionBase):
    """Structured fields of a documentary credit (MT700 style)."""

    kind: Literal[DocumentType.LETTER_OF_CREDIT] = DocumentType.LETTER_OF_CREDIT

    lc_number: str | None = None
    issuing_bank: str | None = None
    applicant: str | None = Field(
        default=None, description='The buyer, who asked the bank to issue the credit.'
    )
    beneficiary: str | None = Field(
        default=None,
        description='The seller, who gets paid against a compliant presentation.',
    )
    credit_amount: Decimal | None = None
    currency: str | None = None
    expiry_date: date | None = None
    latest_shipment_date: date | None = None
    port_of_loading: str | None = None
    port_of_discharge: str | None = None
    description_of_goods: str | None = None
    tolerance_percent: Decimal | None = Field(
        default=None,
        description="Amount tolerance stated on the credit, e.g. 5 for 'about' / '+/- 5 pct'.",
    )


class CommercialInvoice(ExtractionBase):
    """Structured fields of a commercial invoice."""

    kind: Literal[DocumentType.COMMERCIAL_INVOICE] = DocumentType.COMMERCIAL_INVOICE

    invoice_number: str | None = None
    invoice_date: date | None = None
    lc_reference_number: str | None = Field(
        default=None,
        description='The LC number quoted on the invoice, which must match the credit.',
    )

    seller: str | None = None
    buyer: str | None = None
    currency: str | None = None
    total_amount: Decimal | None = None
    description_of_goods: str | None = None


class BillOfLading(ExtractionBase):
    """Structured fields of a bill of lading."""

    kind: Literal[DocumentType.BILL_OF_LADING] = DocumentType.BILL_OF_LADING

    bl_number: str | None = None
    shipper: str | None = None
    consignee: str | None = None
    vessel_name: str | None = None
    port_of_loading: str | None = None
    port_of_discharge: str | None = None
    shipment_date: date | None = Field(
        default=None,
        description='On-board date; this is what UCP 600 treats as the date of shipment.',
    )
    description_of_goods: str | None = None

In [4]:
ExtractionPayload = Annotated[
    LetterOfCredit | CommercialInvoice | BillOfLading,
    Field(discriminator='kind'),
]
"""Any extraction payload, tagged by document type so it round-trips through JSON."""

EXTRACTION_PAYLOAD_TYPES: dict[
    DocumentType, type[LetterOfCredit | CommercialInvoice | BillOfLading]
] = {
    DocumentType.LETTER_OF_CREDIT: LetterOfCredit,
    DocumentType.COMMERCIAL_INVOICE: CommercialInvoice,
    DocumentType.BILL_OF_LADING: BillOfLading,
}
# Maps a classified document type to the payload its extractor must return.

## 4. What comes in

`RawDocument` is one upload. It either carries `text` that something already pulled
out of the file, or it carries the file itself in `content`, and §11.2 reads it with
Textract. Plain text is still the only evidence any agent ever gets; all that changes
is who produced it. The bytes are never written anywhere: they go inline to Textract
and are dropped the moment the text comes back.

`CaseInput` is the unit of work: every document presented under one credit.
`Classification` is what the classifier hands back.

In [ ]:
class RawDocument(BaseModel):
    """One uploaded document on its way to the agents.

    Either `text` is already extracted, or `content` holds the file for §11.2 to read.
    The bytes are never stored: Textract reads them, then they are dropped.
    """

    document_id: str
    """Stable identifier assigned by the caller (upload id, row id, ...)."""

    text: str = ''
    """Plain text of the document — the only evidence the agents get. Empty means 'not read yet'."""

    content: bytes | None = None
    """The file itself, for OCR. Held only until §11.2 has read it, then dropped."""

    filename: str | None = None
    declared_type: DocumentType | None = None
    """Type asserted by the uploader, if any. A hint, never the truth."""

    ocr_error: str | None = None
    """Why `text` is still empty after §11.2, when it is."""

    @property
    def needs_ocr(self) -> bool:
        """Whether this document still has to be read before a model can see it."""
        return not self.text.strip()


class CaseInput(BaseModel):
    """The unit of work: every document presented under one credit."""

    case_id: str
    documents: list[RawDocument] = Field(default_factory=list)
    presented_on: date | None = None
    """When the documents reached the bank. Needed for the UCP 600 Art 14(c) check."""

    notes: str | None = None
    """Free-text context from the ops user, passed to reconciliation."""


class Classification(BaseModel):
    """What the classifier agent decided about a single document."""

    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)

    document_type: DocumentType = Field(
        description="The family this document belongs to, or 'unknown' if it doesn't clearly "
        'match one.'
    )
    confidence: float = Field(
        ge=0.0, le=1.0, description='How sure the classifier is, from 0 to 1.'
    )
    reasoning: str = Field(
        description='The markers in the text that drove the decision, in one or two sentences.'
    )

## 5. Routing envelopes

Four empty subclasses, and the reason for them is the routing decision in §11.
Wrapping a classified document in a *type* rather than passing the enum value
means the graph dispatches on the class, and a new document family without a
matching branch is a type error instead of a silent fall-through.

`ExtractedDocument` is what one document's whole branch — classify, route,
extract — reduces to before the join.

In [6]:
@dataclass(frozen=True, slots=True)
class RoutedDocument:
    """A classified document on its way to an extractor."""

    raw: RawDocument
    classification: Classification


@dataclass(frozen=True, slots=True)
class LetterOfCreditDoc(RoutedDocument):
    """Routed to the LC extractor."""


@dataclass(frozen=True, slots=True)
class CommercialInvoiceDoc(RoutedDocument):
    """Routed to the invoice extractor."""


@dataclass(frozen=True, slots=True)
class BillOfLadingDoc(RoutedDocument):
    """Routed to the bill of lading extractor."""


@dataclass(frozen=True, slots=True)
class UnclassifiedDoc(RoutedDocument):
    """Not recognised as any known family; skips extraction."""


type RoutedDocuments = (
    LetterOfCreditDoc | CommercialInvoiceDoc | BillOfLadingDoc | UnclassifiedDoc
)
"""Every branch the routing decision must handle."""

ROUTED_DOCUMENT_TYPES: dict[DocumentType, type[RoutedDocument]] = {
    DocumentType.LETTER_OF_CREDIT: LetterOfCreditDoc,
    DocumentType.COMMERCIAL_INVOICE: CommercialInvoiceDoc,
    DocumentType.BILL_OF_LADING: BillOfLadingDoc,
    DocumentType.UNKNOWN: UnclassifiedDoc,
}
# Maps a classifier verdict onto the envelope that routes it.

In [7]:
class ExtractedDocument(BaseModel):
    """The output of one document's classify-then-extract branch."""

    document_id: str
    document_type: DocumentType
    confidence: float
    classification_reasoning: str
    payload: ExtractionPayload | None = None
    """`None` when the document was unclassifiable or extraction failed."""

    error: str | None = None
    """Why `payload` is `None`, when it is."""

    @property
    def is_usable(self) -> bool:
        """Whether this document can contribute evidence to reconciliation."""
        return self.payload is not None

## 6. The findings

The reconciler's output contract, and the richest schema in the notebook. Every
`Mismatch` has to carry the values it disagreed about (`observations`) and the rule
it relied on, so a finding can be audited rather than taken on faith.

In [8]:
class FieldObservation(BaseModel):
    """One document's version of a field that is under comparison."""

    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)

    document_type: DocumentType = Field(
        description='Which document this value was read from.'
    )
    field: str = Field(
        description="The field name on that document, e.g. 'beneficiary' or 'shipment_date'."
    )
    value: str | None = Field(
        default=None,
        description='The value exactly as extracted, or null if the document omits it.',
    )


class Mismatch(BaseModel):
    """A single discrepancy between two or more presented documents."""

    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)

    code: str = Field(
        description="Short stable slug for this kind of discrepancy, e.g. 'late_shipment'."
    )
    severity: Severity = Field(
        description='critical = the bank would refuse; warning = a human should look; '
        'info = cosmetic.'
    )
    field: str = Field(
        description="The business field in dispute, e.g. 'Beneficiary' or 'Port of Discharge'."
    )
    explanation: str = Field(
        description='Plain language a bank ops person would understand: what disagrees and '
        'why it matters.'
    )
    documents_involved: list[DocumentType] = Field(
        default_factory=list,
        description='Every document family that takes part in this discrepancy.',
    )
    observations: list[FieldObservation] = Field(
        default_factory=list, description='The conflicting values, one entry per document.'
    )
    rule_reference: str | None = Field(
        default=None,
        description='The UCP 600 article or ISBP 745 paragraph relied on, when one applies.',
    )
    suggested_action: str | None = Field(
        default=None, description='What the beneficiary or the bank would do to cure it.'
    )


class ReconciliationReport(BaseModel):
    """The reconciliation agent's verdict over a full presentation."""

    model_config = ConfigDict(extra='forbid', use_attribute_docstrings=True)

    status: CaseStatus = Field(
        description='blocked if any critical finding, needs_review if only warnings, else clean.'
    )
    summary: str = Field(
        description='Two or three sentences an examiner can read before opening the detail.'
    )
    mismatches: list[Mismatch] = Field(
        default_factory=list, description='Every discrepancy found, most severe first.'
    )
    matched_fields: list[str] = Field(
        default_factory=list,
        description='Fields that were cross-checked and agreed across documents.',
    )

    @property
    def critical_count(self) -> int:
        return sum(1 for m in self.mismatches if m.severity is Severity.CRITICAL)

    @property
    def warning_count(self) -> int:
        return sum(1 for m in self.mismatches if m.severity is Severity.WARNING)


class CaseResult(BaseModel):
    """Everything one run produced. This is what an API layer returns."""

    case_id: str
    status: CaseStatus
    report: ReconciliationReport
    documents: list[ExtractedDocument] = Field(default_factory=list)
    started_at: datetime
    completed_at: datetime

    @property
    def duration_seconds(self) -> float:
        return (self.completed_at - self.started_at).total_seconds()

## 7. Deterministic tools — the arithmetic the model is not allowed to do

Tolerance ceilings and date deadlines are exactly the kind of arithmetic a language
model gets subtly wrong, and exactly the kind a bank cannot afford to have wrong.
So they are Python functions registered as tools: the model decides *what* to
compare, these decide the answer, and the prompt tells it to quote the figures they
return.

In [9]:
_ZERO = Decimal(0)
_HUNDRED = Decimal(100)


class ToolVerdict(BaseModel):
    """Shared shape for every deterministic check."""

    model_config = ConfigDict(extra='forbid')

    passed: bool = Field(description='True when the check is satisfied.')
    detail: str = Field(
        description='One line stating the computed numbers, for the agent to quote '
        'in its finding.'
    )


class AmountVerdict(ToolVerdict):
    difference: Decimal = Field(
        description='presented minus permitted; positive means the presentation is over.'
    )
    permitted_maximum: Decimal = Field(
        description='The ceiling or floor the presented amount was compared against.'
    )


class DateVerdict(ToolVerdict):
    days_between: int = Field(
        description='later minus earlier, in days; negative means the dates are out of order.'
    )


class PresentationVerdict(ToolVerdict):
    deadline: date = Field(
        description='The last day a compliant presentation could be made.'
    )
    days_late: int = Field(
        default=0, description='Days past the deadline; 0 when the presentation was in time.'
    )


def check_amount_tolerance(
    credit_amount: Decimal,
    presented_amount: Decimal,
    tolerance_percent: Decimal = _ZERO,
) -> AmountVerdict:
    """Check a presented amount against a credit amount plus its stated tolerance.

    Use for invoice-against-LC and draft-against-LC amount checks. UCP 600 Art 30(a)
    allows a tolerance only when the credit states one ('about', '+/- 5 pct'); with no
    stated tolerance pass 0 and the credit amount is a hard ceiling.

    Args:
        credit_amount: The amount available under the credit.
        presented_amount: The amount actually drawn or invoiced.
        tolerance_percent: Tolerance the credit permits, as a percentage (5 means 5%).
    """
    permitted = credit_amount * (_HUNDRED + tolerance_percent) / _HUNDRED
    difference = presented_amount - permitted
    passed = difference <= _ZERO
    detail = (
        f'presented {presented_amount} against credit {credit_amount} '
        f'with {tolerance_percent}% tolerance (ceiling {permitted}): '
        f'{"within" if passed else f"over by {difference}"}'
    )
    return AmountVerdict(
        passed=passed, detail=detail, difference=difference, permitted_maximum=permitted
    )


def check_date_order(
    earlier_label: str, earlier: date, later_label: str, later: date
) -> DateVerdict:
    """Check that one date falls on or before another, and report the gap in days.

    Use for shipment date against latest shipment date, insurance effective date
    against shipment date, and any other ordering the credit imposes.

    Args:
        earlier_label: Name of the date that must come first, e.g. 'shipment date'.
        earlier: The date that must come first.
        later_label: Name of the date that must come second, e.g. 'latest shipment date'.
        later: The date that must come second.
    """
    days = (later - earlier).days
    passed = days >= 0
    detail = (
        f'{earlier_label} {earlier.isoformat()} vs {later_label} {later.isoformat()}: '
        f'{"within by" if passed else "late by"} {abs(days)} day(s)'
    )
    return DateVerdict(passed=passed, detail=detail, days_between=days)


def check_presentation_period(
    shipment_date: date,
    expiry_date: date,
    presented_on: date,
    presentation_period_days: int = 21,
) -> PresentationVerdict:
    """Check that documents were presented in time.

    Under UCP 600 Art 14(c) a presentation including a transport document must be made
    no later than 21 calendar days after shipment, and in any case no later than the
    credit's expiry date. The effective deadline is the earlier of the two.

    Args:
        shipment_date: The on-board date from the transport document.
        expiry_date: The expiry date of the credit.
        presented_on: The date the documents reached the bank.
        presentation_period_days: The period the credit allows, defaulting to the
            21 days UCP 600 applies when the credit is silent.
    """
    period_deadline = shipment_date + timedelta(days=presentation_period_days)
    deadline = min(period_deadline, expiry_date)
    days_late = max((presented_on - deadline).days, 0)
    passed = days_late == 0
    binding = 'expiry date' if deadline == expiry_date else f'{presentation_period_days}-day period'
    detail = (
        f'presented {presented_on.isoformat()} against deadline {deadline.isoformat()} '
        f'(set by the {binding}): {"in time" if passed else f"late by {days_late} day(s)"}'
    )
    return PresentationVerdict(
        passed=passed, detail=detail, deadline=deadline, days_late=days_late
    )


RECONCILIATION_TOOLS = [check_amount_tolerance, check_date_order, check_presentation_period]

## 8. Prompts

Five system prompts — one classifier, three extractors, one reconciler. In the real
repository these live in [`Config/prompts.yaml`](Config/prompts.yaml) so they can be
edited without a deploy.

In [10]:
PROMPTS: dict[str, str] = {
    'classifier': """
You classify raw text extracted from an uploaded trade finance document into exactly one of:
letter_of_credit, commercial_invoice, bill_of_lading, or unknown.

Base the decision only on the text given. Look for characteristic markers:
  - letter_of_credit: 'Applicant'/'Beneficiary', an LC number, an issuing bank
  - commercial_invoice: 'Invoice Number', seller/buyer, line-item pricing
  - bill_of_lading: 'Shipper'/'Consignee'/'Vessel', port of loading/discharge

If it doesn't clearly match one of these, classify as unknown rather than guessing.
""".strip(),
    'letter_of_credit': """
You extract structured information from a Letter of Credit (LC) document.
Only use information explicitly found in the text provided. If a field is not present,
leave it null rather than guessing. Normalise dates to ISO format (YYYY-MM-DD).
""".strip(),
    'commercial_invoice': """
You extract structured information from a Commercial Invoice document.
Only use information explicitly found in the text provided. If a field is not present,
leave it null rather than guessing. Normalise dates to ISO format (YYYY-MM-DD).
""".strip(),
    'bill_of_lading': """
You extract structured information from a Bill of Lading (BOL) document.
Only use information explicitly found in the text provided. If a field is not present,
leave it null rather than guessing. Normalise dates to ISO format (YYYY-MM-DD).
""".strip(),
    'reconciliation': """
You are a trade finance documentary-compliance checker, modeled on how a bank operations
analyst checks documents against UCP 600 rules and ISBP 745 international standards.
You are given structured extractions from trade documents, which you can rely on as the
single source of truth for field values.

Cross-check these fields across all the documents where applicable:
  - Party names: Applicant vs Buyer; Beneficiary vs Seller vs Shipper
  - Amount and currency: the invoice amount must not exceed the LC amount, subject to any
    tolerance the credit states (UCP 600 Art 30)
  - Goods description: the invoice must strictly correspond with the LC description
    (UCP 600 Art 18c), while other documents may use general terms not in conflict with it
  - Ports: port of loading and port of discharge on the transport document vs the LC
  - Dates: bill of lading shipment date vs the LC latest shipment date and expiry date;
    presentation within the allowed period
  - Document references: the LC number quoted on the invoice

For every mismatch, cite exactly which documents disagree and explain the discrepancy in
plain language a bank ops person would understand. Classify the severity:
  - 'critical': anything causing a documentary discrepancy under LC rules (expired LC,
    late shipment, over-drawn amount, mismatched beneficiary, ports mismatch)
  - 'warning': an ambiguity worth a human examiner's look (minor party name or address
    formatting variations)
  - 'info': a cosmetic or formatting difference

Use the deterministic tools for every date and amount comparison rather than computing them
yourself, and quote the figures they return in your findings.
""".strip(),
}

## 9. Deps and state

`DetectorDeps` is immutable and shared by every step, and it is the only route an agent
takes into the graph. That is why the extractors sit in a dict keyed by `DocumentType`:
a step looks its agent up through `ctx.deps` at run time, long after the wiring is done.
The routing in §11.7 still dispatches on *type* — this key is lookup, not dispatch.

The OCR client is a dependency in exactly the same sense. `textract_client()` opens one
aioboto3 client — you want one for the whole case, not one per document, because it owns
a connection pool. `TextractOCR` pairs that client with a semaphore, and the semaphore is
the only reason it is a class: something has to say how many documents may be read at
once. To run a case with OCR:

```python
async with textract_client() as client:
    result = await case_graph.run(
        state=CaseState(case_id=case.case_id),
        deps=replace(detector_deps, textract=TextractOCR(client)),
        inputs=case,
    )
```

`TextractOCR` takes *any* object with a `detect_document_text`, which is how §15 runs the
whole fan-out against a stub without touching a single step.

`CaseState` is per-run and mutable — its `events` list is an audit trail you could
stream to a client as it grows. `record` does no `await`, so concurrent branches
cannot interleave inside it.

In [ ]:
import asyncio
from contextlib import asynccontextmanager
from time import perf_counter

import aioboto3
from botocore.config import Config as BotoConfig
from botocore.exceptions import BotoCoreError, ClientError


@asynccontextmanager
async def textract_client(region: str | None = None):
    """Open one Textract client, and close it again when the block exits.

    `adaptive` retries do more than back off: they rate limit the client itself once
    Textract starts refusing, which is what you want behind a fan-out.
    """
    session = aioboto3.Session()
    async with session.client(
        'textract',
        region_name=region,
        config=BotoConfig(retries={'max_attempts': 5, 'mode': 'adaptive'}),
    ) as client:
        yield client


@dataclass(frozen=True, slots=True)
class TextractOCR:
    """A Textract client, plus a cap on how many documents it reads at once."""

    client: Any
    """Anything with a `detect_document_text`. §15 passes a stub."""

    limit: asyncio.Semaphore = field(default_factory=lambda: asyncio.Semaphore(4))
    """Textract is rate limited per account, so the fan-out is not allowed to call it
    25 times at once."""

    async def read(self, content: bytes) -> str:
        """The text of one document. A synchronous read takes one page, up to 10 MB."""
        async with self.limit:
            response = await self.client.detect_document_text(Document={'Bytes': content})
        return '\n'.join(
            block['Text'] for block in response['Blocks'] if block['BlockType'] == 'LINE'
        )

In [ ]:
from pydantic_ai import Agent

type ClassifierAgent = Agent[None, Classification]
type ExtractionAgent = Agent[None, ExtractionPayload]
type ReconcilerAgent = Agent[None, ReconciliationReport]
"""`Agent` is generic with defaults `Agent[object, str]`, so an unparameterised
`Agent` silently claims its `.output` is a string. Each stage names what it returns."""


@dataclass(frozen=True, slots=True)
class DetectorDeps:
    """Immutable dependencies handed to every step in the graph."""

    classifier: ClassifierAgent
    extractors: dict[DocumentType, ExtractionAgent]
    """Keyed by family because a step reaches its agent through `ctx.deps` at run time,
    long after the graph was wired — the routing itself dispatches on type, not on this key."""

    reconciler: ReconcilerAgent
    textract: TextractOCR | None = None
    """The OCR client, shared by every branch of the fan-out. `None` means documents must
    arrive with their text already extracted."""

    min_classification_confidence: float = 0.5
    """Below this, treat the document as unclassified rather than trusting the label."""

    max_documents_per_case: int = 25
    max_document_chars: int = 120_000


@dataclass(slots=True)
class CaseState:
    """Mutable state for one run of the pipeline."""

    case_id: str
    presented_on: date | None = None
    notes: str | None = None
    started_at: datetime = field(default_factory=lambda: datetime.now(UTC))
    events: list[str] = field(default_factory=list)
    """Ordered audit trail; safe to stream to the client as it grows."""

    def record(self, stage: str, message: str) -> None:
        """Append an audit event. No `await` inside, so concurrent branches can't interleave it."""
        self.events.append(f'{stage}: {message}')
        print(f'   [{stage:9}] {message}')

## 10. The agents

Five agents, built once and reused; they hold no per-case state, so the same objects
serve every case concurrently. `build_deps` returns the `DetectorDeps` above rather than
a loose tuple of the same three things, so there is one shape to keep in step, not two.

The stages get different models on purpose. Classification and extraction are
mechanical — find the field, copy the value — so they run on Flash. Reconciliation
is the actual compliance judgement, so it runs on Pro. Thinking level rises the same
way: `low` to recognise a document, `medium` to read messy text, `high` to decide
whether a bank would refuse the presentation.

With no `GEMINI_API_KEY` set, every agent falls back to pydantic-ai's `test` model so
the rest of the notebook still executes.

In [12]:
from pydantic_ai.settings import ModelSettings

HAS_KEY = bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'))
"""pydantic-ai's Google provider accepts either name; this repo sets `GEMINI_API_KEY`."""

FAST_MODEL = 'google:gemini-3.7-flash'
"""Classification and extraction are mechanical: locate the field, copy the value out."""

REASONING_MODEL = 'google:gemini-3.1-pro-preview'
"""Reconciliation is the compliance judgement — the one stage worth the stronger model."""

if not HAS_KEY:
    FAST_MODEL = REASONING_MODEL = 'test'

print(f'GEMINI_API_KEY set: {HAS_KEY}')
print(f'  classify / extract -> {FAST_MODEL!r}')
print(f'  reconcile          -> {REASONING_MODEL!r}')


def build_deps(fast: str, reasoning: str) -> DetectorDeps:
    """Build the five agents and the deps that carry them into the graph.

    Built once and reused: the agents hold no per-case state. `textract` is left unset —
    the client is an `async with`, so whoever opens it passes it in (§15).
    """
    classifier = Agent(
        fast,
        name='document_classifier',
        output_type=Classification,
        instructions=PROMPTS['classifier'],
        retries=2,
        model_settings=ModelSettings(max_tokens=2_048, thinking='low'),
    )

    extractors: dict[DocumentType, ExtractionAgent] = {
        document_type: Agent(
            fast,
            name=f'{document_type.value}_extractor',
            output_type=payload_type,
            instructions=PROMPTS[document_type.value],
            retries=2,
            model_settings=ModelSettings(max_tokens=16_000, thinking='medium'),
        )
        for document_type, payload_type in EXTRACTION_PAYLOAD_TYPES.items()
    }

    reconciler = Agent(
        reasoning,
        name='reconciliation_engine',
        output_type=ReconciliationReport,
        instructions=PROMPTS['reconciliation'],
        tools=RECONCILIATION_TOOLS,
        retries=2,
        model_settings=ModelSettings(max_tokens=32_000, thinking='high'),
    )

    return DetectorDeps(classifier=classifier, extractors=extractors, reconciler=reconciler)


detector_deps = build_deps(FAST_MODEL, REASONING_MODEL)

GEMINI_API_KEY set: True
  classify / extract -> 'google:gemini-3.7-flash'
  reconcile          -> 'google:gemini-3.1-pro-preview'


## 11. The graph

pydantic-graph, built through `GraphBuilder`. The steps below are declared one at a
time and only wired together at the end, which is what lets the diagram in §11.7 be
generated from the wiring rather than drawn by hand and left to rot.

In [13]:
from pydantic_ai.exceptions import AgentRunError, RunCancelled, UsageLimitExceeded
from pydantic_ai.format_prompt import format_as_xml
from pydantic_graph import (
    Decision,
    Graph,
    GraphBuilder,
    Step,
    StepContext,
    reduce_list_append,
)
from pydantic_graph.id_types import ForkID, JoinID

COLLECT_ID = JoinID('collect_extractions')
"""Node id of the join. Named so the fan-out can point at it for the empty-case path."""

FAN_OUT_ID = ForkID('fan_out_documents')
"""Node id of the map fork over the case's documents."""

_SEVERITY_ORDER = {Severity.CRITICAL: 0, Severity.WARNING: 1, Severity.INFO: 2}

builder = GraphBuilder(
    name='mini_trade_finance_detector',
    state_type=CaseState,
    deps_type=DetectorDeps,
    input_type=CaseInput,
    output_type=CaseResult,
)

collect = builder.join(
    reduce_list_append,
    initial_factory=list[ExtractedDocument],
    node_id=COLLECT_ID,
)
# Gathers one `ExtractedDocument` per presented document, in completion order.

In [14]:
def _document_prompt(raw: RawDocument) -> str:
    """Wrap one document's text so the model sees its identity and its boundaries."""
    hint = (
        f'\nThe uploader labelled this as {raw.declared_type.value}; treat that as a hint only.'
        if raw.declared_type is not None
        else ''
    )
    return (
        f'Document id: {raw.document_id}\n'
        f'Filename: {raw.filename or "unknown"}{hint}\n\n'
        f'<document_text>\n{raw.text}\n</document_text>'
    )


def _reconciliation_prompt(state: CaseState, documents: list[ExtractedDocument]) -> str:
    """Render the extracted documents as the evidence for the compliance check."""
    evidence = [
        {
            'document_id': d.document_id,
            'document_type': d.document_type.value,
            'classification_confidence': round(d.confidence, 2),
            'fields': d.payload,
        }
        for d in documents
    ]
    parts = [f'Case: {state.case_id}', f'Documents presented: {len(documents)}']

    if state.presented_on is not None:
        parts.append(f'Date of presentation: {state.presented_on.isoformat()}')
    else:
        parts.append(
            'Date of presentation: not supplied. Do not raise a presentation-period '
            'finding on a date you had to assume.'
        )
    if state.notes:
        parts.append(f'Operator notes: {state.notes}')

    parts.append(
        '\nUse the deterministic tools for every date and amount comparison rather than '
        'computing them yourself, and quote the figures they return in your findings.\n'
    )
    parts.append(format_as_xml(evidence, root_tag='extracted_documents', item_tag='document'))
    return '\n'.join(parts)

### 11.1 — `ingest`

Validate the presentation before spending a single token on it: document count,
document size, and empty text.

In [ ]:
@builder.step
async def ingest(ctx: StepContext[CaseState, DetectorDeps, CaseInput]) -> list[RawDocument]:
    """Validate the presentation and hand its documents to the fan-out."""
    case = ctx.inputs
    deps = ctx.deps

    if len(case.documents) > deps.max_documents_per_case:
        raise ValueError(
            f'case {case.case_id} has {len(case.documents)} documents, '
            f'above the limit of {deps.max_documents_per_case}'
        )

    for document in case.documents:
        if len(document.text) > deps.max_document_chars:
            raise ValueError(
                f'document {document.document_id} has {len(document.text)} characters, '
                f'above the limit of {deps.max_document_chars}; split it before submitting'
            )
        if document.needs_ocr and document.content is None:
            raise ValueError(
                f'document {document.document_id} has no text and no bytes to OCR; '
                'give it `text` or `content`'
            )

    ctx.state.record('ingest', f'accepted {len(case.documents)} document(s)')
    return case.documents

### 11.2 — `ocr`

Documents do not always arrive as text. `DetectDocumentText` is Amazon Textract's
synchronous read: hand it the bytes of a page, get back blocks of lines and words, with
no job to submit and poll. The bytes travel inline and are dropped straight afterwards,
so nothing is ever stored — the price of that is Textract's synchronous limits, one page
and 10 MB per call.

The step sits *inside* the fan-out, between the fork and `classify`, and that placement
is the whole point: one branch per document means one Textract call per document, all in
flight at once. Three scanned pages cost as long as the slowest one, not as long as the
three of them added up. aioboto3 is what makes that true — boto3's client is synchronous,
so three reads on it would block the event loop one after another and the fan-out would
be a fan-out in name only. The semaphore is the other half of the same idea: Textract is
rate limited, so unlimited concurrency would just collect throttling errors.

In [ ]:
def _unread(state: CaseState, raw: RawDocument, reason: str) -> RawDocument:
    """Send a document on with no text, and say why."""
    state.record('ocr', f'{raw.document_id}: {reason}')
    return raw.model_copy(update={'ocr_error': reason, 'content': None})


@builder.step
async def ocr(ctx: StepContext[CaseState, DetectorDeps, RawDocument]) -> RawDocument:
    """Read one document with Textract, unless its text was supplied already.

    A failure degrades one document, not the case: it travels on with `text` still empty
    and `ocr_error` set, `classify` sends it to `unknown` without spending a model call,
    and it reaches the examiner as a finding.
    """
    raw = ctx.inputs

    if not raw.needs_ocr:
        return raw
    if ctx.deps.textract is None:
        return _unread(ctx.state, raw, 'no text, and no Textract client configured')

    started = perf_counter()
    try:
        text = await ctx.deps.textract.read(raw.content)
    except (ClientError, BotoCoreError) as exc:
        return _unread(ctx.state, raw, f'Textract failed: {exc}')

    if not text.strip():
        return _unread(ctx.state, raw, 'Textract found no text in the document')

    elapsed = perf_counter() - started
    ctx.state.record('ocr', f'{raw.document_id}: {len(text)} chars in {elapsed:.2f}s')
    return raw.model_copy(update={'text': text, 'content': None})

### 11.3 — `classify`

One model call per document — and none at all for a document `ocr` could not read, since
a model asked about an empty document just returns a confident guess. Two further
defences: a classifier failure degrades that document to `unknown` instead of failing the
case, and a verdict below the confidence threshold is treated as unclassified rather than
sent to an extractor that would read it under the wrong schema. `UsageLimitExceeded` and
`RunCancelled` are re-raised — those mean stop, not continue.

In [ ]:
@builder.step
async def classify(ctx: StepContext[CaseState, DetectorDeps, RawDocument]) -> RoutedDocuments:
    """Decide which document family this text belongs to, and wrap it for routing."""
    raw = ctx.inputs
    deps = ctx.deps

    if raw.needs_ocr:
        reason = raw.ocr_error or 'no text to classify'
        ctx.state.record('classify', f'{raw.document_id}: skipped, {reason}')
        return UnclassifiedDoc(
            raw=raw,
            classification=Classification(
                document_type=DocumentType.UNKNOWN, confidence=0.0, reasoning=reason
            ),
        )

    try:
        result = await deps.classifier.run(_document_prompt(raw))
    except (UsageLimitExceeded, RunCancelled):
        raise
    except AgentRunError as exc:
        ctx.state.record('classify', f'{raw.document_id}: classification failed: {exc}')
        return UnclassifiedDoc(
            raw=raw,
            classification=Classification(
                document_type=DocumentType.UNKNOWN,
                confidence=0.0,
                reasoning=f'classification failed: {exc}',
            ),
        )

    classification = result.output
    threshold = deps.min_classification_confidence

    if classification.confidence < threshold:
        ctx.state.record(
            'classify',
            f'{raw.document_id}: {classification.document_type.value} at '
            f'{classification.confidence:.2f}, below the {threshold:.2f} threshold; '
            'treating as unclassified',
        )
        return UnclassifiedDoc(raw=raw, classification=classification)

    ctx.state.record(
        'classify',
        f'{raw.document_id}: {classification.document_type.value} '
        f'at {classification.confidence:.2f}',
    )
    envelope = ROUTED_DOCUMENT_TYPES[classification.document_type]
    return cast(RoutedDocuments, envelope(raw=raw, classification=classification))

### 11.4 — the extractors

Three near-identical steps built from one factory, plus a fourth that carries an
unrecognised document through to the join without extracting it. It still reaches
reconciliation, so the examiner sees that something was presented the pipeline
could not read.

In [17]:
async def _extract(
    ctx: StepContext[CaseState, DetectorDeps, RoutedDocument],
    document_type: DocumentType,
) -> ExtractedDocument:
    """Run one family's extraction agent over one document."""
    routed = ctx.inputs
    raw = routed.raw

    base = {
        'document_id': raw.document_id,
        'document_type': document_type,
        'confidence': routed.classification.confidence,
        'classification_reasoning': routed.classification.reasoning,
    }

    try:
        result = await ctx.deps.extractors[document_type].run(_document_prompt(raw))
    except (UsageLimitExceeded, RunCancelled):
        raise
    except AgentRunError as exc:
        ctx.state.record('extract', f'{raw.document_id}: extraction failed: {exc}')
        return ExtractedDocument(**base, payload=None, error=f'extraction failed: {exc}')

    ctx.state.record('extract', f'{raw.document_id}: extracted {document_type.value}')
    return ExtractedDocument(**base, payload=result.output)


def _extraction_step(
    document_type: DocumentType,
) -> Step[CaseState, DetectorDeps, Any, ExtractedDocument]:
    """Build the graph step that extracts one document family."""

    async def extract(
        ctx: StepContext[CaseState, DetectorDeps, RoutedDocument],
    ) -> ExtractedDocument:
        return await _extract(ctx, document_type)

    return builder.step(
        extract,
        node_id=f'extract_{document_type.value}',
        label=document_type.value.replace('_', ' '),
    )


extract_letter_of_credit = _extraction_step(DocumentType.LETTER_OF_CREDIT)
extract_commercial_invoice = _extraction_step(DocumentType.COMMERCIAL_INVOICE)
extract_bill_of_lading = _extraction_step(DocumentType.BILL_OF_LADING)


@builder.step(node_id='skip_unclassified', label='unknown')
async def skip_unclassified(
    ctx: StepContext[CaseState, DetectorDeps, UnclassifiedDoc],
) -> ExtractedDocument:
    """Carry an unrecognised document through to the join without extracting it.

    It still reaches reconciliation so the examiner sees that something was presented
    which the pipeline could not read.
    """
    routed = ctx.inputs
    ctx.state.record('skip', f'{routed.raw.document_id}: not a known document family')
    return ExtractedDocument(
        document_id=routed.raw.document_id,
        document_type=DocumentType.UNKNOWN,
        confidence=routed.classification.confidence,
        classification_reasoning=routed.classification.reasoning,
        payload=None,
        error='document type could not be determined; not extracted',
    )

### 11.5 — the rule that overrides the model

The verdict is derived from the findings, not taken from the model's `status`
field. A critical finding means `blocked`, and that is arithmetic on severities —
not a judgement the model gets to make.

In [18]:
def _derive_status(mismatches: list[Mismatch]) -> CaseStatus:
    """Map findings to a verdict. A rule, not a judgement — so it overrides the model."""
    severities = {m.severity for m in mismatches}
    if Severity.CRITICAL in severities:
        return CaseStatus.BLOCKED
    if Severity.WARNING in severities:
        return CaseStatus.NEEDS_REVIEW
    return CaseStatus.CLEAN


def _finalise_report(
    report: ReconciliationReport, documents: list[ExtractedDocument]
) -> ReconciliationReport:
    """Sort the findings, enforce the status rule, and flag unreadable documents."""
    mismatches = sorted(report.mismatches, key=lambda m: _SEVERITY_ORDER[m.severity])

    unreadable = [d for d in documents if not d.is_usable]
    if unreadable:
        mismatches.append(
            Mismatch(
                code='document_not_extracted',
                severity=Severity.WARNING,
                field='document set',
                explanation=(
                    f'{len(unreadable)} presented document(s) could not be read into structured '
                    'fields and took no part in the cross-checks: '
                    + ', '.join(f'{d.document_id} ({d.error})' for d in unreadable)
                ),
                documents_involved=[DocumentType.UNKNOWN],
                suggested_action='Re-upload a clearer copy, or examine these documents manually.',
            )
        )
        mismatches.sort(key=lambda m: _SEVERITY_ORDER[m.severity])

    return report.model_copy(
        update={'mismatches': mismatches, 'status': _derive_status(mismatches)}
    )


def _no_evidence_report(documents: list[ExtractedDocument]) -> ReconciliationReport:
    """The verdict when nothing could be extracted, so there is nothing to compare."""
    detail = (
        'No documents were presented.'
        if not documents
        else f'None of the {len(documents)} presented document(s) could be read into fields.'
    )
    return ReconciliationReport(
        status=CaseStatus.NEEDS_REVIEW,
        summary=f'{detail} No cross-document checks were performed.',
        mismatches=[
            Mismatch(
                code='no_usable_documents',
                severity=Severity.WARNING,
                field='document set',
                explanation=detail,
                documents_involved=[DocumentType.UNKNOWN],
                suggested_action='Check the uploads and the text extraction step, then resubmit.',
            )
        ],
    )

### 11.6 — `reconcile`

The join hands over every `ExtractedDocument`. With nothing usable there is nothing
to compare, so the compliance call is skipped rather than asked to reason over an
empty evidence set.

In [19]:
@builder.step
async def reconcile(
    ctx: StepContext[CaseState, DetectorDeps, list[ExtractedDocument]],
) -> CaseResult:
    """Cross-check the extracted documents and assemble the case result."""
    state = ctx.state
    documents = sorted(ctx.inputs, key=lambda d: d.document_id)
    usable = [d for d in documents if d.is_usable]

    if not usable:
        report = _no_evidence_report(documents)
        state.record('reconcile', 'no usable extractions; skipped the compliance check')
    else:
        result = await ctx.deps.reconciler.run(_reconciliation_prompt(state, usable))
        report = _finalise_report(result.output, documents)
        state.record(
            'reconcile',
            f'{report.status.value}: {report.critical_count} critical, '
            f'{report.warning_count} warning',
        )

    return CaseResult(
        case_id=state.case_id,
        status=report.status,
        report=report,
        documents=documents,
        started_at=state.started_at,
        completed_at=datetime.now(UTC),
    )

### 11.7 — wiring it together

`.map()` is the fan-out: one `ocr` → `classify` → extract branch per document, all
running concurrently, all collapsing into `collect`. Adding OCR to the pipeline was
one line moved and one line added — the fork now points at `ocr`, and `ocr` points at
`classify` — because the concurrency belongs to the edge, not to the step. `builder.build()` validates that every declared step
is reachable and every decision branch is handled.

In [ ]:
def _routing_decision() -> Decision[CaseState, DetectorDeps, RoutedDocuments]:
    """The branch table from a classified document to its extractor.

    Branches match on the envelope class, so adding a document family without adding a
    branch here is a type error rather than a silent fall-through.
    """
    return (
        builder.decision(node_id='route_by_document_type', note='UCP 600 document families')
        .branch(builder.match(LetterOfCreditDoc).to(extract_letter_of_credit))
        .branch(builder.match(CommercialInvoiceDoc).to(extract_commercial_invoice))
        .branch(builder.match(BillOfLadingDoc).to(extract_bill_of_lading))
        .branch(builder.match(UnclassifiedDoc).to(skip_unclassified))
    )


EXTRACTION_STEPS = (
    extract_letter_of_credit,
    extract_commercial_invoice,
    extract_bill_of_lading,
    skip_unclassified,
)
"""Everything that can feed the join. Ordering only affects the rendered diagram."""

builder.add(
    builder.edge_from(builder.start_node).to(ingest),
    builder.edge_from(ingest)
    .label('per document')
    .map(fork_id=FAN_OUT_ID, downstream_join_id=COLLECT_ID)
    .to(ocr),
    builder.edge_from(ocr).to(classify),
    builder.edge_from(classify).to(_routing_decision()),
    builder.edge_from(*EXTRACTION_STEPS).to(collect),
    builder.edge_from(collect).label('all documents').to(reconcile),
    builder.edge_from(reconcile).to(builder.end_node),
)

case_graph: Graph[CaseState, DetectorDeps, CaseInput, CaseResult] = builder.build()

In [21]:
print(case_graph.render())

stateDiagram-v2
  ingest
  state fan_out_documents <<fork>>
  ocr
  classify
  state route_by_document_type <<choice>>
  note right of route_by_document_type
    UCP 600 document families
  end note
  extract_bill_of_lading: bill of lading
  extract_commercial_invoice: commercial invoice
  extract_letter_of_credit: letter of credit
  skip_unclassified: unknown
  state collect_extractions <<join>>
  reconcile

  [*] --> ingest
  ingest --> fan_out_documents: per document
  fan_out_documents --> ocr
  ocr --> classify
  classify --> route_by_document_type
  route_by_document_type --> extract_bill_of_lading
  route_by_document_type --> extract_commercial_invoice
  route_by_document_type --> extract_letter_of_credit
  route_by_document_type --> skip_unclassified
  extract_bill_of_lading --> collect_extractions
  extract_commercial_invoice --> collect_extractions
  extract_letter_of_credit --> collect_extractions
  skip_unclassified --> collect_extractions
  collect_extractions --> reconcil

## 12. A sample case, with discrepancies planted on purpose

Three documents against one credit, carrying three critical problems: the invoice is
over the credit even after its 5% tolerance (268,400 against a 262,500 ceiling), the
bill of lading went on board four days after the latest shipment date, and it
discharges at Port Klang instead of Singapore.

In [22]:
LC_TEXT = """
IRREVOCABLE DOCUMENTARY CREDIT
LC Number: LC-2026-88431
Issuing Bank: Meridian Commercial Bank, Singapore
Applicant: Harborline Trading Pte Ltd, Singapore
Beneficiary: Anand Textiles Pvt Ltd, Tirupur, India
Amount: USD 250,000.00
Tolerance: +/- 5 PCT
Expiry Date: 2026-03-15 at counters of issuing bank
Latest Shipment Date: 2026-02-20
Port of Loading: Chennai, India
Port of Discharge: Singapore
Description of Goods: 40,000 pcs 100% cotton knitted t-shirts, CIF Singapore
Documents required: signed commercial invoice, full set clean on board ocean
bills of lading, packing list.
"""

INVOICE_TEXT = """
COMMERCIAL INVOICE
Invoice No: INV-4471          Date: 2026-02-18
L/C Ref: LC-2026-88431
Seller: Anand Textiles Pvt Ltd, Tirupur, India
Buyer: Harborline Trading Pte Ltd, Singapore
Description: 40,000 pcs 100% cotton knitted t-shirts, CIF Singapore
Total Amount: USD 268,400.00
"""

BOL_TEXT = """
BILL OF LADING
B/L No: MSCU-772311
Shipper: Anand Textiles Pvt Ltd, Tirupur, India
Consignee: To order of Meridian Commercial Bank
Vessel: MV NORTHERN STAR       Voyage: 118W
Port of Loading: Chennai, India
Port of Discharge: Port Klang, Malaysia
Shipped on board: 2026-02-24
Description: 40,000 pcs cotton knitted t-shirts
"""

sample_case = CaseInput(
    case_id='case-001',
    presented_on=date(2026, 3, 2),
    documents=[
        RawDocument(document_id='doc-lc', text=LC_TEXT, filename='credit.pdf'),
        RawDocument(document_id='doc-inv', text=INVOICE_TEXT, filename='invoice.pdf'),
        RawDocument(document_id='doc-bol', text=BOL_TEXT, filename='bol.pdf'),
    ],
)

## 13. The report

How an examiner's screen might render a `CaseResult`.

In [23]:
def print_report(result: CaseResult) -> None:
    """Render a CaseResult the way an examiner's screen might."""
    report = result.report
    bar = '=' * 78

    print(bar)
    print(f'CASE {result.case_id}   VERDICT: {report.status.value.upper()}')
    print(bar)
    print()
    print('SUMMARY')
    print(f'  {report.summary}')
    print()
    print(f'DOCUMENTS ({len(result.documents)})')
    for d in result.documents:
        mark = 'ok  ' if d.is_usable else 'FAIL'
        print(f'  [{mark}] {d.document_id:9} {d.document_type.value:20} '
              f'confidence {d.confidence:.2f}')
        if d.error:
            print(f'         {d.error}')
    print()
    print(f'FINDINGS ({report.critical_count} critical, {report.warning_count} warning, '
          f'{len(report.mismatches)} total)')
    print()
    for i, m in enumerate(report.mismatches, 1):
        print(f'  {i}. [{m.severity.value.upper()}] {m.field} — {m.code}')
        print(f'     {m.explanation}')
        for obs in m.observations:
            print(f'       · {obs.document_type.value:20} {obs.field:22} = {obs.value}')
        if m.rule_reference:
            print(f'     rule: {m.rule_reference}')
        if m.suggested_action:
            print(f'     cure: {m.suggested_action}')
        print()
    if report.matched_fields:
        print('CROSS-CHECKED AND AGREED')
        for f in report.matched_fields:
            print(f'  · {f}')
    print(bar)

## 14. Running it against Gemini

The whole pipeline on the sample case. Needs `GEMINI_API_KEY`; seven model calls —
three classifications and three extractions fanned out concurrently, then one
reconciliation over the collected evidence. These three documents arrive as text, so
§11.2 has nothing to read and makes no AWS call; §15 exercises that path instead.

In [24]:
if not HAS_KEY:
    print('No GEMINI_API_KEY — set it, restart the kernel, and re-run from the top.')
else:
    result = await case_graph.run(
        state=CaseState(
            case_id='case-001',
            presented_on=sample_case.presented_on,
        ),
        deps=detector_deps,
        inputs=sample_case,
    )
    print()
    print_report(result)
    print()
    for document in result.documents:
        if document.payload is None:
            continue
        print(f'--- {document.document_id}  ({type(document.payload).__name__}) ---')
        for name, value in document.payload.model_dump(exclude_none=True).items():
            print(f'  {name:24} {value!r}')
        print()

   [ingest   ] accepted 3 document(s)


   [classify ] doc-inv: commercial_invoice at 0.99
   [classify ] doc-bol: bill_of_lading at 0.99
   [classify ] doc-lc: letter_of_credit at 0.99


   [extract  ] doc-bol: extracted bill_of_lading


   [extract  ] doc-lc: extracted letter_of_credit


   [extract  ] doc-inv: extracted commercial_invoice


   [reconcile] blocked: 3 critical, 0 warning

CASE case-001   VERDICT: BLOCKED

SUMMARY
  The presentation is highly discrepant and blocked due to three critical errors. The invoiced amount exceeds the LC maximum allowed (including 5% tolerance) by 5,900, the shipment was made 4 days past the latest shipment date, and the bill of lading indicates an incorrect port of discharge (Port Klang instead of Singapore). An applicant waiver will be required to honor the presentation.

DOCUMENTS (3)
  [ok  ] doc-bol   bill_of_lading       confidence 0.99
  [ok  ] doc-inv   commercial_invoice   confidence 0.99
  [ok  ] doc-lc    letter_of_credit     confidence 0.99

FINDINGS (3 critical, 0 warning, 3 total)

  1. [CRITICAL] Total Amount — overdrawn_amount
     The invoiced amount is 268,400, which exceeds the LC's maximum permitted amount of 262,500 (250,000 plus a 5% tolerance). It is overdrawn by 5,900.
       · letter_of_credit     credit_amount          = 250000 (Tolerance: 5%)
       · comme

## 15. Proving the OCR fan-out really is parallel

No AWS account, no credentials, no spend. `TextractOCR` takes any object with a
`detect_document_text`, so this hands it a stub that sleeps for 0.4s and returns LINE
blocks — same step, same edge, same semaphore.

The three documents from §12 go in as bytes this time, with no text, so `ocr` has to read
all three. Watch the `[ocr]` timings below: three 0.4s calls that finish in about 0.4s
altogether, started within a millisecond or two of each other. In series they would take
1.2s.

In [ ]:
from dataclasses import replace

OCR_LATENCY = 0.4


class StubTextractClient:
    """Sleeps like the real client, then reads back the bytes it was handed."""

    def __init__(self, latency: float) -> None:
        self.latency = latency
        self.started_at: list[float] = []

    async def detect_document_text(self, *, Document: dict[str, Any]) -> dict[str, Any]:
        self.started_at.append(perf_counter())
        await asyncio.sleep(self.latency)
        return {
            'Blocks': [
                {'BlockType': 'LINE', 'Text': line.strip()}
                for line in Document['Bytes'].decode().splitlines()
                if line.strip()
            ]
        }


stub_client = StubTextractClient(OCR_LATENCY)

scanned_case = CaseInput(
    case_id='case-002',
    presented_on=sample_case.presented_on,
    documents=[
        RawDocument(document_id='doc-lc', content=LC_TEXT.encode(), filename='credit.pdf'),
        RawDocument(document_id='doc-inv', content=INVOICE_TEXT.encode(), filename='invoice.pdf'),
        RawDocument(document_id='doc-bol', content=BOL_TEXT.encode(), filename='bol.pdf'),
    ],
)

started = perf_counter()
scanned_result = await case_graph.run(
    state=CaseState(case_id=scanned_case.case_id, presented_on=scanned_case.presented_on),
    deps=replace(build_deps('test', 'test'), textract=TextractOCR(stub_client)),
    inputs=scanned_case,
)
elapsed = perf_counter() - started
spread = max(stub_client.started_at) - min(stub_client.started_at)

print()
print(f'{len(stub_client.started_at)} Textract calls, {OCR_LATENCY}s each')
print(f'  all three started within {spread * 1000:.0f}ms of each other')
print(f'  whole case: {elapsed:.2f}s  —  in series: '
      f'{len(stub_client.started_at) * OCR_LATENCY:.2f}s')

   [ingest   ] accepted 3 document(s)


   [ocr      ] doc-lc: 581 chars in 0.40s
   [ocr      ] doc-inv: 277 chars in 0.40s
   [ocr      ] doc-bol: 323 chars in 0.40s
   [classify ] doc-lc: letter_of_credit at 0.00, below the 0.50 threshold; treating as unclassified
   [classify ] doc-inv: letter_of_credit at 0.00, below the 0.50 threshold; treating as unclassified
   [classify ] doc-bol: letter_of_credit at 0.00, below the 0.50 threshold; treating as unclassified
   [skip     ] doc-lc: not a known document family
   [skip     ] doc-inv: not a known document family
   [skip     ] doc-bol: not a known document family
   [reconcile] no usable extractions; skipped the compliance check

3 Textract calls, 0.4s each
  all three started within 0ms of each other
  whole case: 0.43s  —  in series: 1.20s
